# Multi-Agent Demo: Agent Calling Agent

Minimal proof of concept showing an **outer agent** calling an **inner agent** via `DATA_AGENT_RUN` wrapped in a UDF tool.

**Flow:** User → Outer Agent → UDF tool → `DATA_AGENT_RUN(inner_agent)` → response bubbles back up

In [ ]:
%%sql -r setup_result
CREATE DATABASE IF NOT EXISTS MULTI_AGENT_DEMO_DB;
CREATE SCHEMA IF NOT EXISTS MULTI_AGENT_DEMO_DB.DEMO;

In [ ]:
%%sql -r inner_agent_result
CREATE OR REPLACE AGENT MULTI_AGENT_DEMO_DB.DEMO.INNER_AGENT
  COMMENT = 'A simple joke-telling agent'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    system: 'You are a joke teller. When asked for a joke on a topic, reply with exactly one short joke about that topic. Keep it under 50 words.'
  $$;

In [ ]:
CREATE OR REPLACE PROCEDURE MULTI_AGENT_DEMO_DB.DEMO.CALL_INNER_AGENT(topic VARCHAR)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import json

def run(session, topic: str) -> str:
    request_body = json.dumps({
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"Tell me a joke about: {topic}"
                    }
                ]
            }
        ],
        "stream": False
    })
    escaped = request_body.replace("'", "''")
    sql = f"SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('MULTI_AGENT_DEMO_DB.DEMO.INNER_AGENT', '{escaped}') AS response"
    result = session.sql(sql).collect()
    response = json.loads(result[0]['RESPONSE'])
    return response
$$;

In [ ]:
%%sql -r outer_agent_result
CREATE OR REPLACE AGENT MULTI_AGENT_DEMO_DB.DEMO.OUTER_AGENT
  COMMENT = 'Orchestrator that delegates joke requests to the inner agent'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    system: 'You are an assistant. If the user asks for a joke, use the get_joke tool to fetch one and tell them you are doing it. Return the joke to the user as-is. If you used the get_joke tool, you must credit the inner agent with inventing the joke'
    orchestration: 'Always use the get_joke tool when the user wants a joke. Never invent your own joke.'

  tools:
    - tool_spec:
        type: generic
        name: get_joke
        description: 'Fetches a joke about a given topic from a specialist joke agent.'
        input_schema:
          type: object
          properties:
            topic:
              type: string
              description: 'The topic for the joke, e.g. cats, programming, etc.'
          required:
            - topic

  tool_resources:
    get_joke:
      type: procedure
      execution_environment:
        type: warehouse
        warehouse: APP_WH
      identifier: MULTI_AGENT_DEMO_DB.DEMO.CALL_INNER_AGENT
  $$;

In [ ]:
%%sql -r agent_response
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'MULTI_AGENT_DEMO_DB.DEMO.OUTER_AGENT',
  $${
    "messages": [
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "Tell me a joke about programming"
          }
        ]
      }
    ],
    "stream": false
  }$$
) AS response;

In [ ]:
%%sql -r cleanup_result
DROP AGENT IF EXISTS MULTI_AGENT_DEMO_DB.DEMO.OUTER_AGENT;
DROP AGENT IF EXISTS MULTI_AGENT_DEMO_DB.DEMO.INNER_AGENT;
DROP PROCEDURE IF EXISTS MULTI_AGENT_DEMO_DB.DEMO.CALL_INNER_AGENT(VARCHAR);
DROP SCHEMA IF EXISTS MULTI_AGENT_DEMO_DB.DEMO;
DROP DATABASE IF EXISTS MULTI_AGENT_DEMO_DB;